# PS-C1 — Tunable H₂ VQE across Processor A/BTeam Quantum Mind · Based on the supplied H2.ipynb reduced two-qubit VQE. The source notebook is preserved as H2_source.ipynb; this notebook adds Processor A/B routing benchmarks.Edit config.json to tune shots, optimizer budget, initial layout, and noise mode. Noise rates remain off until officially confirmed.

In [ ]:
# If needed, install the pinned-compatible dependencies in requirements.txt
%pip install -r requirements.txt

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from src.problem import hamiltonian, exact_ground_energy, EXACT_TOTAL_HARTREE
from src.processors import load_processor
from src.visualization import draw_topology
from src.experiment import run

config = json.loads(Path('config.json').read_text())
print('R (Angstrom):', config['geometry_angstrom'])
print('Exact reduced-Hamiltonian ground energy (Hartree):', exact_ground_energy())
print('Challenge reference (Hartree):', EXACT_TOTAL_HARTREE)

## Fixed processor architectures
The maps below come from the supplied simulation configuration PDF. Processor B uses all nine qubits.

In [ ]:
fig_a = draw_topology(load_processor('processor_A'))
fig_b = draw_topology(load_processor('processor_B'))
plt.show()

## Run the A/B experimentEach energy evaluation uses the H2.ipynb ansatz and finite-shot Pauli measurements. The circuits are separately transpiled against each physical coupling map. Results are saved under results/.

In [ ]:
table, histories = run()
display(table)

In [ ]:
for name, history in histories.items():
    plt.plot([row[1] for row in history], marker='.', label=name)
plt.axhline(exact_ground_energy(), color='black', linestyle='--', label='exact diagonalization')
plt.xlabel('Objective evaluation'); plt.ylabel('Energy (Hartree)')
plt.title('Shot-based COBYLA VQE convergence'); plt.legend(); plt.grid(alpha=.2); plt.show()

## InterpretationCompare energy error, transpiled depth, CX count, and SWAP count. Routing overhead depends on the chosen initial_layout. The supplied H2 Hamiltonian coefficients are for 0.735 Å (near the challenge 0.74 Å); regenerate integrals for exactly 0.740 Å.Finite-shot estimates can fall below the exact eigenvalue due to sampling variance; the underlying expectation still obeys the variational principle.